# CommonLID paper tables

This notebook loads the per-(model, dataset) `summary.json` +
`predictions.jsonl` files produced by `commonlid run` (LLMs are evaluated
by passing `--model dspy:<provider>/<model>` to the same `run` command)
and regenerates the core analysis tables and plots used in the paper.

**Inputs:** a results directory with the layout
`{results_dir}/{dataset_id}/{model_id}/{summary.json,predictions.jsonl}`.

**Outputs (in-notebook):**
- Macro / micro F1 table per (model, dataset)
- Per-language F1 pivot table
- False-positive-rate table restricted to each model's supported languages
- Simple comparison plots

To generate input files:
```bash
commonlid run --model GlotLID --model cld2 --dataset commonlid --dataset udhr --output-dir ./results
commonlid generate-support-matrix --out ./results/support_matrix.csv  # optional
```

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from commonlid.evaluation.results import load_summary
from commonlid.metrics.fpr import false_positive_rate
from commonlid.metrics.support_matrix import load_support_matrix

RESULTS_DIR = Path("../results")
SUPPORT_MATRIX_CSV = RESULTS_DIR / "support_matrix.csv"  # optional; set to None to skip

## 1. Flatten every `summary.json` into a per-language DataFrame

In [ ]:
def load_results(results_dir: Path) -> pd.DataFrame:
    rows: list[dict] = []
    for path in sorted(results_dir.rglob("summary.json")):
        s = load_summary(path)
        for language, m in s["per_language"].items():
            rows.append({
                "dataset_id": s["dataset_id"],
                "model_id": s["model_id"],
                "language": language,
                "gt_count": m["gt_count"],
                "predictions": m["predictions"],
                "correct": m["correct"],
                "precision": m["precision"],
                "recall": m["recall"],
                "f1": m["f1"],
                # Schema v2: paper-style "gold-only" view is the headline.
                "macro_f1": s["macro"]["f1_gold_only"],
                "micro_f1": s["micro"]["f1_gold_only"],
                "samples_per_second": s["samples_per_second"],
            })
    return pd.DataFrame(rows)


df = load_results(RESULTS_DIR)
df.head()

## 2. Macro / micro F1 per (model, dataset)

In [ ]:
per_run = (
    df
    .groupby(["dataset_id", "model_id"])
    .agg(
        macro_f1=("macro_f1", "first"),
        micro_f1=("micro_f1", "first"),
        samples_per_second=("samples_per_second", "first"),
        n_languages=("language", "nunique"),
    )
    .reset_index()
)
per_run

In [ ]:
macro_f1_pivot = per_run.pivot(index="model_id", columns="dataset_id", values="macro_f1").round(4)
macro_f1_pivot

## 3. Per-language F1 pivot (fix a dataset to inspect)

In [ ]:
DATASET = "commonlid"  # change to any dataset_id present in your results

subset = df[df["dataset_id"] == DATASET]
per_language = subset.pivot(index="language", columns="model_id", values="f1").fillna(0.0)
per_language.head(20)

## 4. Per-supported-language F1 (filter by the support matrix)

Use the CSV written by `commonlid generate-support-matrix` to restrict each
model's mean F1 to the languages it actually claims to support.

In [ ]:
if SUPPORT_MATRIX_CSV is not None and SUPPORT_MATRIX_CSV.exists():
    support = load_support_matrix(SUPPORT_MATRIX_CSV)

    def _supported_mean(row: pd.DataFrame) -> pd.Series:
        supported = support.get(row.name[1], set())
        in_support = row[row["language"].isin(supported)]
        return pd.Series({
            "mean_f1_supported": in_support["f1"].mean() if len(in_support) else float("nan"),
            "n_langs_supported_evaluated": len(in_support),
        })

    supported_summary = df.groupby(["dataset_id", "model_id"]).apply(_supported_mean).reset_index()
    supported_summary
else:
    print("Skipping: no support matrix available. Run `commonlid generate-support-matrix`.")

## 5. False-positive rate across non-target languages

For each (model, dataset, language), the fraction of *other* languages'
samples that were labelled as `language`. Uses the raw predictions from
`predictions.jsonl`, so you can drill into per-sample confusions.

In [ ]:
def fpr_table(results_dir: Path, target_language: str, dataset_id: str) -> pd.DataFrame:
    import json

    rows = []
    for preds_path in sorted((results_dir / dataset_id).rglob("predictions.jsonl")):
        model_id = preds_path.parent.name
        ytrue, ypred = [], []
        with preds_path.open() as f:
            for line in f:
                record = json.loads(line)
                ytrue.append(record["gold"])
                ypred.append(record["pred"])
        rows.append({
            "model_id": model_id,
            "language": target_language,
            "fpr": false_positive_rate(ytrue, ypred, language=target_language),
        })
    return pd.DataFrame(rows)


# Example: how often does each model mistakenly predict 'eng' on non-English inputs?
fpr_table(RESULTS_DIR, target_language="eng", dataset_id=DATASET)

## 6. Plot: macro F1 by (model, dataset)

In [ ]:
ax = macro_f1_pivot.plot.bar(figsize=(10, 4))
ax.set_ylabel("macro F1")
ax.set_ylim(0, 1)
ax.set_title("Macro F1 by model x dataset")
ax.legend(title="dataset", loc="center left", bbox_to_anchor=(1.02, 0.5))